# 04 — Forecasting: Federal Accounts

Forecast quarterly obligated spending at the **federal account level** using Prophet, SARIMA, and XGBoost.

- **Data**: 2,236 federal accounts, FY2017–2024 (up to 31 quarters each)
- **Train**: FY2017–2022 | **Test**: FY2023–2024
- **Individual models**: Top 20 accounts (Prophet + SARIMA per account)
- **Cross-account model**: Single XGBoost on all 2,236 accounts

**Key results (total federal spending):**
| Model   | MAE      | RMSE     | MAPE   |
|---------|----------|----------|--------|
| SARIMA  | $144B    | $157B    | **3.8%** ✓ |
| XGBoost | $329B    | $396B    | 5.8%   |
| Prophet | $1,902B  | $2,328B  | 30.5%  |

**Per-account top performers (Prophet MAPE):** SSA Old-Age Trust Fund 2.8% · Health Care Trust Funds 3.4% · Disability Trust Fund 4.1% · Civil Service Retirement 4.7%

**Unpredictable accounts (both models fail):** U.S. Coronavirus Payments · Business Loans (SBA PPP) · Unemployment Trust Fund · Federal Direct Student Loan

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT         = Path('..').resolve()
CLEAN_A      = ROOT / 'pipeline-a-hierarchical' / 'data' / 'cleaned'
FORECAST_DIR = ROOT / 'pipeline-a-hierarchical' / 'data' / 'forecasts'
FORECAST_DIR.mkdir(exist_ok=True)

def trillions(x, _): return f'${x/1e12:.2f}T'
def billions(x, _):  return f'${x/1e9:.1f}B'

def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1e6   # exclude near-zero accounts from MAPE
    if mask.sum() == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mp   = mape(np.array(y_true), np.array(y_pred))
    print(f'{label:22}  MAE=${mae/1e9:.2f}B  RMSE=${rmse/1e9:.2f}B  MAPE={mp:.1f}%')
    return {'model': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mp}

TRAIN_END  = 2022
TEST_START = 2023
TOP_N      = 20   # individual models for top-N accounts

print('Setup done.')

One important difference from the budget functions and agency notebooks: the `mape()` function here excludes near-zero values (below $1M) from the percentage error calculation. Federal accounts include many accounts that were active for one year due to a specific appropriation and then went to zero — those near-zero denominators would make MAPE meaningless. The $1M threshold keeps the metric focused on accounts with substantive spending.

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv(CLEAN_A / 'federal_accounts_ALL_FY.csv',
                 dtype={'budget_function_id': str, 'budget_subfunction_id': str})

quarter_to_month = {1: (10, -1), 2: (1, 0), 3: (4, 0), 4: (7, 0)}
def fy_q_to_date(row):
    month, yr_offset = quarter_to_month[row['quarter']]
    return pd.Timestamp(year=int(row['fy']) + yr_offset, month=month, day=1)

df['ds']          = df.apply(fy_q_to_date, axis=1)
df['covid']       = df['fy'].isin([2020, 2021]).astype(int)
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

# Short display names for plots
df['acct_short'] = df['federal_account_name'].str.split(',').str[0].str.strip()

print(f'Rows: {len(df):,}  Accounts: {df["federal_account_id"].nunique():,}')
print(f'FY range: {df["fy"].min()}–{df["fy"].max()}')
print(df[['fy','quarter','ds','federal_account_id','acct_short','obligated_amount']].head(4).to_string(index=False))

Federal account names are long and include agency suffixes (e.g., "Grants to States for Medicaid, Centers for Medicare and Medicaid Services, Health and Human Services"). The `acct_short` column extracts just the first comma-separated segment for plot labels. The full name is kept in `federal_account_name` for the saved output files.

In [ ]:
# Total federal spending series
total = (df.groupby(['fy', 'quarter', 'ds', 'covid'])
           .agg(obligated_amount=('obligated_amount', 'sum'))
           .reset_index().sort_values('ds'))

# Top N accounts by cumulative spend
top_ids   = df.groupby('federal_account_id')['obligated_amount'].sum().nlargest(TOP_N).index.tolist()
acct_names = df.drop_duplicates('federal_account_id').set_index('federal_account_id')['acct_short'].to_dict()

train_total = total[total['fy'] <= TRAIN_END]
test_total  = total[total['fy'] >= TEST_START]

print(f'Total series — Train: {len(train_total)} rows  Test: {len(test_total)} rows')
print(f'\nTop {TOP_N} accounts by cumulative spend:')
for i, aid in enumerate(top_ids, 1):
    tot = df[df['federal_account_id'] == aid]['obligated_amount'].sum()
    print(f'  {i:2}. {aid:6}  {acct_names[aid][:55]:<55}  ${tot/1e12:.2f}T')

The federal accounts dataset is an extreme long-tail distribution. The top 20 accounts out of 2,236 include the largest mandatory spending vehicles in the US budget:

| # | Account | Total (FY2017–2024) |
|---|---------|---------------------|
| 1 | SSA Old-Age & Survivors Trust Fund | $19.9T |
| 2 | Interest on the Public Debt | $13.2T |
| 3 | Grants to States for Medicaid | $11.2T |
| 4 | Medicare SMI Trust Fund (Part B) | $8.2T |
| 5 | Payments to Health Care Trust Funds | $8.0T |
| 12 | U.S. Coronavirus Payments | $2.2T |
| 14 | Business Loans Program (SBA PPP) | $1.9T |

Accounts 12 and 14 in this list are COVID-era one-time programs — their high cumulative totals are due entirely to FY2020–2021 payouts, and their spending in FY2022–2024 is near zero. These will be flagged as "high uncertainty" in the dashboard rather than forecasted with false precision.

## 3. Prophet — Total Series

In [ ]:
def run_prophet(train_df, test_df, series_name='total'):
    prophet_train = train_df.rename(columns={'obligated_amount': 'y'})[['ds','y','covid']]
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05
    )
    m.add_regressor('covid')
    m.fit(prophet_train)
    future = m.make_future_dataframe(periods=len(test_df), freq='QS-OCT')
    future['covid'] = future['ds'].dt.year.isin([2019, 2020]).astype(int)
    forecast = m.predict(future)
    pred = forecast.tail(len(test_df))['yhat'].values
    true = test_df['obligated_amount'].values
    result = metrics(true, pred, label='Prophet')
    result.update({'series': series_name, 'pred': pred, 'true': true, 'ds': test_df['ds'].values})
    return result, m, forecast

prophet_result, prophet_model, prophet_forecast = run_prophet(train_total, test_total, 'total')

Prophet on the total federal accounts series — this is structurally the same series as the total in notebooks 02 and 03 (all federal obligations per quarter). The result should confirm the pattern seen there: systematic underestimation due to the post-COVID spending floor increase, with the gap widening through FY2024.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=4, lw=1.5,
        label='Actual', color='steelblue')
ax.fill_between(prophet_forecast['ds'],
                prophet_forecast['yhat_lower'], prophet_forecast['yhat_upper'],
                alpha=0.15, color='orange', label='95% confidence')
ax.plot(prophet_forecast['ds'], prophet_forecast['yhat'], '--', lw=1.5,
        color='orange', label='Prophet forecast')
ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Prophet — Total Federal Accounts Spending Forecast')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

comparison_p = pd.DataFrame({
    'Quarter':      [f'FY{int(r.fy)} Q{int(r.quarter)}' for _, r in test_total.iterrows()],
    'Actual ($B)':  (test_total['obligated_amount'].values / 1e9).round(1),
    'Prophet ($B)': (prophet_result['pred'] / 1e9).round(1),
    'Error ($B)':   ((prophet_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print('Prophet — quarter-by-quarter comparison:')
print(comparison_p.to_string(index=False))

**Prophet Total Results — MAPE 30.5% | MAE $1,902B | RMSE $2,328B**

The third consecutive notebook where Prophet shows the same pattern: systematic underestimation that deepens through FY2024, reaching −$4,801B by Q4 FY2024 (−50% of actual). The errors here are nearly identical to the agency notebook (30.6% MAPE), confirming that the issue is not how the data is sliced but the structural post-COVID spending shift that all three hierarchical views share. The federal accounts total, agency total, and budget functions total are all representations of the same underlying obligation stream — so the same Prophet failure mode applies to all three.

## 4. SARIMA — Total Series

In [ ]:
def run_sarima(train_df, test_df, order=(1,1,1), seasonal_order=(1,1,0,4), series_name='total'):
    train_y    = train_df.set_index('ds')['obligated_amount']
    train_exog = train_df.set_index('ds')[['covid']]
    test_exog  = test_df.set_index('ds')[['covid']]
    model  = SARIMAX(train_y, exog=train_exog, order=order, seasonal_order=seasonal_order,
                     enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False)
    pred   = fitted.forecast(steps=len(test_df), exog=test_exog)
    true   = test_df['obligated_amount'].values
    result = metrics(true, pred.values, label='SARIMA')
    result.update({'series': series_name, 'pred': pred.values, 'true': true,
                   'ds': test_df['ds'].values})
    return result, fitted

sarima_result, sarima_fitted = run_sarima(train_total, test_total, series_name='total')
print(f'\nAIC: {sarima_fitted.aic:.1f}   BIC: {sarima_fitted.bic:.1f}')

SARIMAX(1,1,1)(1,1,0,4) on the total federal accounts series. Since this is the same aggregate time series modeled in notebooks 02 and 03, the result should be consistent — SARIMA has proven reliable at 3.8% MAPE on this data structure regardless of how the underlying accounts are organized.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=4, lw=1.5,
        label='Actual', color='steelblue')
ax.plot(test_total['ds'], sarima_result['pred'], 's--', ms=5, lw=1.5,
        label='SARIMA forecast', color='green')
ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('SARIMA — Total Federal Accounts Spending Forecast')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

comparison_s = pd.DataFrame({
    'Quarter':     [f'FY{int(r.fy)} Q{int(r.quarter)}' for _, r in test_total.iterrows()],
    'Actual ($B)': (test_total['obligated_amount'].values / 1e9).round(1),
    'SARIMA ($B)': (sarima_result['pred'] / 1e9).round(1),
    'Error ($B)':  ((sarima_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print('SARIMA — quarter-by-quarter comparison:')
print(comparison_s.to_string(index=False))

**SARIMA Total Results — MAPE 3.8% | MAE $144B | RMSE $157B | AIC 759.4**

Identical result to notebooks 02 and 03 — 3.8% MAPE, AIC=759.4, BIC=762.3, and the same quarter-by-quarter error pattern (+$150B, −$233B, −$127B, −$44B, +$210B, +$182B, −$58B, −$145B). This is mathematically expected: the total federal accounts series is the same data as the total budget functions and total agency series. The result confirms that SARIMA's dominance on the aggregate is stable across all three perspectives, and that the individual-level model choices (Prophet vs SARIMA per-account) do not affect the total-level conclusion.

## 5. XGBoost — All 2,236 Accounts

In [ ]:
df_xgb = df.sort_values(['federal_account_id', 'ds']).copy()

df_xgb['lag_1']     = df_xgb.groupby('federal_account_id')['obligated_amount'].shift(1)
df_xgb['lag_4']     = df_xgb.groupby('federal_account_id')['obligated_amount'].shift(4)
df_xgb['lag_8']     = df_xgb.groupby('federal_account_id')['obligated_amount'].shift(8)
df_xgb['roll4_mean'] = df_xgb.groupby('federal_account_id')['obligated_amount'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=2).mean())

# Encode account id and budget function
le_acct = LabelEncoder()
le_func = LabelEncoder()
df_xgb['acct_enc'] = le_acct.fit_transform(df_xgb['federal_account_id'])
df_xgb['func_enc'] = le_func.fit_transform(df_xgb['budget_function_id'].astype(str))

df_xgb = df_xgb.dropna(subset=['lag_1', 'lag_4', 'lag_8'])

FEATURES = ['acct_enc','func_enc','fy','quarter','quarter_sin','quarter_cos',
            'covid','lag_1','lag_4','lag_8','roll4_mean']
TARGET   = 'obligated_amount'

train_xgb = df_xgb[df_xgb['fy'] <= TRAIN_END]
test_xgb  = df_xgb[df_xgb['fy'] >= TEST_START]

print(f'XGBoost train: {len(train_xgb):,} rows   test: {len(test_xgb):,} rows')
print(f'Features: {FEATURES}')

Two encoded identifiers are used here — `acct_enc` (the specific federal account) and `func_enc` (its budget function). Adding budget function gives XGBoost a grouping signal one level above the account, which helps when an account has sparse history: the model can borrow patterns from other accounts in the same function. With 2,236 accounts across only 8 fiscal years, cross-series pattern sharing is more important than in the agency notebook (116 agencies over 8 years).

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(train_xgb[FEATURES], train_xgb[TARGET],
              eval_set=[(test_xgb[FEATURES], test_xgb[TARGET])],
              verbose=False)

xgb_pred = xgb_model.predict(test_xgb[FEATURES])
xgb_result = metrics(test_xgb[TARGET].values, xgb_pred, label='XGBoost')

importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nFeature importance:')
print(importance.round(4).to_string())

**XGBoost Feature Importance — roll4_mean (44.4%) leads, not lag_4**

A notable shift from the budget functions and agency notebooks: the **4-quarter rolling mean dominates at 44.4%**, while `lag_4` fell to just 4.1%. The reason is the heterogeneity of federal accounts — with 2,236 series ranging from huge trust funds to tiny program accounts with spotty data, the rolling mean is a more robust signal than the single year-ago quarter. Many accounts have irregular histories (new programs starting mid-decade, COVID accounts going from zero to billions and back), so the smoothed average is more reliable than any single lagged value. `lag_8` (16.4%) and `lag_1` (12.2%) contribute meaningfully. `acct_enc` and `func_enc` each score 3.1–3.2% — noticeably higher than `agency_enc` (0.6%) in notebook 03, meaning account identity is a somewhat useful signal across 2,236 distinct programs.

In [ ]:
# Sum predictions to total and plot
test_xgb_copy = test_xgb.copy()
test_xgb_copy['xgb_pred'] = xgb_pred
xgb_total = test_xgb_copy.groupby('ds')[['obligated_amount','xgb_pred']].sum().reset_index()
train_actual = df_xgb[df_xgb['fy'] <= TRAIN_END].groupby('ds')['obligated_amount'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_actual['ds'], train_actual['obligated_amount'], 'o-', ms=3, lw=1.5,
        color='steelblue', label='Actual (train)')
ax.plot(xgb_total['ds'], xgb_total['obligated_amount'], 'o-', ms=3, lw=1.5,
        color='steelblue', alpha=0.4, label='Actual (test)')
ax.plot(xgb_total['ds'], xgb_total['xgb_pred'], 's--', ms=5, lw=1.5,
        color='purple', label='XGBoost forecast')
ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('XGBoost — Total Federal Accounts Spending (Sum Across 2,236 Accounts)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

xgb_total_result = metrics(xgb_total['obligated_amount'].values,
                           xgb_total['xgb_pred'].values, label='XGB-Total')

**XGBoost Total Results — MAPE 5.8% | MAE $329B | RMSE $396B**

Slightly worse than the agency notebook (5.4% MAPE) despite having 19× more series. The RMSE increase from $278B to $396B reflects larger individual account errors not fully cancelling when summed — likely driven by the COVID accounts (U.S. Coronavirus Payments, Business Loans) that have near-zero test spending but non-zero predictions, adding systematic downward error in aggregate. Excluding the COVID accounts from the total would likely bring the XGBoost total MAPE closer to 4–5%. Nevertheless, 5.8% MAPE on the sum of 2,236 accounts is a strong result for a single model with no per-account fitting.

## 6. Per-Account Forecasts — Top 20 (Prophet + SARIMA)

In [ ]:
acct_results = []

for aid in top_ids:
    aname = acct_names.get(aid, str(aid))
    sub   = df[df['federal_account_id'] == aid].sort_values('ds')
    tr    = sub[sub['fy'] <= TRAIN_END]
    te    = sub[sub['fy'] >= TEST_START]

    if len(tr) < 8 or len(te) == 0:
        print(f'  Skip {aname[:40]}: insufficient data')
        continue

    # Prophet
    try:
        r_p, _, _ = run_prophet(tr, te, series_name=str(aid))
        r_p['acct_name'] = aname
        acct_results.append(r_p)
    except Exception as e:
        print(f'  Prophet FAIL {aname[:40]}: {e}')

    # SARIMA
    try:
        r_s, _ = run_sarima(tr, te, series_name=str(aid))
        r_s['acct_name'] = aname
        acct_results.append(r_s)
    except Exception as e:
        print(f'  SARIMA FAIL {aname[:40]}: {e}')

print(f'\nCompleted {len(acct_results)} model runs across {TOP_N} accounts.')

40 models total — Prophet and SARIMA for each of the top 20 accounts. SARIMA diverged catastrophically on several accounts: SSA Old-Age Trust Fund (MAPE 6.1 billion%), Payments to Health Care Trust Funds (2.9 million%), Civil Service Retirement (36,365%), and Business Loans / Student Loans (>10,000%). These are the most extreme numerical instabilities seen across the three forecasting notebooks. Prophet completed all 20 accounts successfully and was the clear winner for most trust fund and entitlement accounts.

In [ ]:
rows = []
for r in acct_results:
    rows.append({
        'Account':   r.get('acct_name', r['series'])[:45],
        'Model':     r['model'],
        'MAE ($B)':  round(r['MAE'] / 1e9, 2),
        'RMSE ($B)': round(r['RMSE'] / 1e9, 2),
        'MAPE (%)':  round(r['MAPE'], 1)
    })

results_df = pd.DataFrame(rows)
print(results_df.sort_values(['Account','Model']).to_string(index=False))

**Per-Account Results — Trust funds are highly predictable; COVID accounts are not**

**Best performers (Prophet MAPE):**
- SSA Old-Age & Survivors Trust Fund: **2.8%** — most predictable account in the dataset
- Payments to Health Care Trust Funds: **3.4%**
- Federal Disability Insurance Trust Fund: **4.1%**
- Civil Service Retirement & Disability Fund: **4.7%**
- Grants to States for Medicaid: **7.4%**
- Payments to Military Retirement Fund: **8.1%** (SARIMA 7.1% — one of the few cases SARIMA wins)
- DoD Working Capital Fund: **8.8%**
- Federal Hospital Insurance Trust Fund: Prophet 15.7% vs **SARIMA 9.1%** (SARIMA wins here)
- Interest on the Public Debt: **9.8%**
- Refunding Internal Revenue Collections: Prophet 36.2% vs **SARIMA 23.3%** (SARIMA wins)

**SARIMA wins on**: Federal Hospital Insurance (Medicare Part A), Military Retirement, and Tax Refunds — accounts with highly regular, non-volatile cash flows that SARIMA's AR structure tracks well.

**Completely unpredictable (both models fail):**
- U.S. Coronavirus Payments: Prophet 301,000%, SARIMA 140,534% — these payments went from $0 to $2.2T during COVID and are near zero now; no historical pattern can predict this
- Business Loans (SBA PPP): Prophet 99,860%, SARIMA 237,749%
- Federal Direct Student Loan: Prophet 266%, SARIMA 11,004% — Biden-era forgiveness attempts created erratic swings
- Unemployment Trust Fund: Prophet 409%, SARIMA 1,210% — COVID UI surge was unprecedented

**Dashboard flag**: The 4 unpredictable accounts should be marked as "data available, forecast unreliable" rather than shown with misleading confidence intervals.

In [ ]:
# 2×2 grid: top 4 accounts — Prophet vs SARIMA
top4 = top_ids[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, aid in zip(axes.flat, top4):
    aname = acct_names.get(aid, str(aid))
    sub   = df[df['federal_account_id'] == aid].sort_values('ds')

    ax.plot(sub['ds'], sub['obligated_amount'], 'o-', ms=3, lw=1.5,
            label='Actual', color='steelblue')

    p_res = next((r for r in acct_results if r['series'] == str(aid) and r['model'] == 'Prophet'), None)
    s_res = next((r for r in acct_results if r['series'] == str(aid) and r['model'] == 'SARIMA'), None)

    if p_res:
        ax.plot(p_res['ds'], p_res['pred'], 's--', ms=4, lw=1.2,
                label=f'Prophet ({p_res["MAPE"]:.1f}%)', color='orange')
    if s_res:
        mape_label = f'{s_res["MAPE"]:.1f}%' if s_res['MAPE'] < 1e6 else 'diverged'
        ax.plot(s_res['ds'], s_res['pred'], '^--', ms=4, lw=1.2,
                label=f'SARIMA ({mape_label})', color='green')

    ax.axvline(pd.Timestamp('2023-10-01'), color='red', lw=0.8, ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(billions))
    ax.set_title(f'{aname[:45]}', fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Prophet vs SARIMA — Top 4 Federal Accounts (Test Period FY2023–2024)', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**Top 4 accounts — SSA Old-Age, Interest on Debt, Medicaid, Medicare SMI**

- **SSA Old-Age Trust Fund** ($19.9T): Prophet **2.8% MAPE** — the single most accurate individual account forecast in the entire project. Social Security retirement payments grow smoothly with demographics and COLA adjustments; Prophet captured this trend almost perfectly. SARIMA diverged (MAPE 6.1 billion%).
- **Interest on the Public Debt** ($13.2T): Prophet **9.8%**, SARIMA 15.1% — Prophet wins. The debt interest series accelerated sharply in FY2022–2024 as the Fed raised rates; Prophet's flexible trend handled this better than SARIMA's fixed differencing structure.
- **Grants to States for Medicaid** ($11.2T): Prophet **7.4%**, SARIMA 70.8% — Prophet wins decisively. Medicaid grants follow enrollment and match rates set by formula; SARIMA's failure here reflects the COVID-era Medicaid enrollment surge that distorted its seasonality estimates.
- **Medicare SMI Trust Fund Part B** ($8.2T): Prophet **10.8%**, SARIMA 16.1% — both models had moderate success; Prophet slightly better. This account covers physician and outpatient services, which grew steadily but saw COVID-related disruption in FY2020.

## 7. Model Comparison

In [ ]:
total_comparison = pd.DataFrame([
    {'Model': 'Prophet',
     'MAE ($B)':  round(prophet_result['MAE']/1e9, 2),
     'RMSE ($B)': round(prophet_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(prophet_result['MAPE'], 1)},
    {'Model': 'SARIMA',
     'MAE ($B)':  round(sarima_result['MAE']/1e9, 2),
     'RMSE ($B)': round(sarima_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(sarima_result['MAPE'], 1)},
    {'Model': 'XGBoost',
     'MAE ($B)':  round(xgb_total_result['MAE']/1e9, 2),
     'RMSE ($B)': round(xgb_total_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(xgb_total_result['MAPE'], 1)},
])
print('=== Total Federal Accounts — Model Comparison ===')
print(total_comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['orange', 'green', 'purple']
for ax, col in zip(axes, ['MAE ($B)', 'RMSE ($B)', 'MAPE (%)']):
    ax.bar(total_comparison['Model'], total_comparison[col], color=colors)
    ax.set_title(col)
    for i, v in enumerate(total_comparison[col]):
        ax.text(i, v * 1.01, str(v), ha='center', fontsize=9)
plt.suptitle('Federal Accounts Forecasting — Model Comparison', y=1.02)
plt.tight_layout()
plt.show()

**Final Scorecard — SARIMA total wins again; Prophet best for most individual accounts; SARIMA wins on 3 accounts**

| Model   | MAE      | RMSE     | MAPE   |
|---------|----------|----------|--------|
| SARIMA  | $144B    | $157B    | **3.8%** ✓ total |
| XGBoost | $329B    | $396B    | 5.8%   |
| Prophet | $1,902B  | $2,328B  | 30.5%  |

**Cross-notebook pattern now fully established across 02, 03, and 04:**
- SARIMA dominates at the aggregate (total) level with 3.8% MAPE consistently
- Prophet is the most reliable per-series model (wins on 14 of 20 accounts with valid comparison)
- SARIMA wins per-series on 3 specific accounts: Medicare Part A (Hospital Insurance), Military Retirement Fund, and Tax Refunds
- 4 accounts are structurally unpredictable from historical data alone (COVID programs, student loans, unemployment)
- XGBoost provides solid total-level accuracy (5.8%) with no per-account fitting — best scalable fallback

**Dashboard model assignment for top 20 accounts:**
- Use **SARIMA** for: Federal Hospital Insurance, Military Retirement, Tax Refunds
- Use **Prophet** for: all other trust funds, entitlements, and stable programs
- Flag as **"high uncertainty"**: U.S. Coronavirus Payments, Business Loans (PPP), Unemployment Trust Fund, Federal Direct Student Loan

## 8. Save Forecasts

In [ ]:
# 1. Total series predictions
total_preds = pd.DataFrame({
    'ds':      test_total['ds'].values,
    'fy':      test_total['fy'].values,
    'quarter': test_total['quarter'].values,
    'actual':  test_total['obligated_amount'].values,
    'prophet': prophet_result['pred'],
    'sarima':  sarima_result['pred'],
    'xgboost': xgb_total['xgb_pred'].values,
})
total_preds.to_csv(FORECAST_DIR / 'federal_accounts_total_predictions.csv', index=False)

# 2. Per-account predictions (top 20)
acct_pred_rows = []
for r in acct_results:
    for ds, pred, true in zip(r['ds'], r['pred'], r['true']):
        acct_pred_rows.append({
            'ds': ds, 'federal_account_id': r['series'],
            'acct_name': r.get('acct_name', ''),
            'model': r['model'], 'actual': true, 'predicted': pred
        })
pd.DataFrame(acct_pred_rows).to_csv(
    FORECAST_DIR / 'federal_accounts_per_account_predictions.csv', index=False)

# 3. Model metrics
total_comparison.to_csv(FORECAST_DIR / 'federal_accounts_model_metrics.csv', index=False)

print('Saved to', FORECAST_DIR)
for f in sorted(FORECAST_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

Three files saved — consistent with the naming convention from notebooks 02 and 03. The per-account file covers the top 20 accounts only; the dashboard will display these with their model-specific MAPE so users can judge confidence. The total predictions file enables the dashboard to show all three models side-by-side on the aggregate federal accounts spending chart.